In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402, F401
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
from matplotlib.colors import ListedColormap, BoundaryNorm  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    SingleDataMetadata,
    MusicTypeVariants,
    ConditionVariants,
)
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
pd.set_option("display.max_rows", 200)
print("Setup complete.")

# Raw Dataset Inspection — Integrity & Completeness

This notebook inspects the **metadata-level properties** of a selected raw dataset,
*before* any preprocessing. It is meant to catch data-collection problems early.

It reports:
* Dataset size and per-participant recording counts
* Condition / music-type balance
* A **presence matrix** of every expected `participant × condition × music_type` cell
* **Duplicate trials** — the same combination recorded more than once
* **Missing trials** — expected combinations with no recording
* **Unexpected trials** — recordings that don't match any expected combination
* Recordings flagged in the excluded-participants list

> **Parameter to tweak:** `EXPERIMENT` in the *Configuration* cell below.

## Configuration

In [ ]:
# ── Dataset to inspect ────────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR  # e.g. ExperimentNames.PSILO_MUSIC

# Music types each experiment is expected to contain. Used to build the set of
# expected trials (so a music type that is missing for *everyone* is still caught).
EXPERIMENT_MUSIC_TYPES = {
    ExperimentNames.PSILO_MUSIC: [
        MusicTypeVariants.CLASSICAL,
        MusicTypeVariants.PSYTRANCE,
    ],
    ExperimentNames.ASSR: [MusicTypeVariants.ASSR],
}

# ── Plot saving ───────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "raw_dataset_inspection"
    / EXPERIMENT.value
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Inspecting experiment: {EXPERIMENT.value}")
print(f"Plots will be saved to: {PLOTS_DIR}")

## Dataset Initialisation

In [ ]:
dataset_handler = DatasetHandler(
    EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
dataset_metadata = dataset_handler.dataset_metadata
participant_map = dataset_handler.dataset_parser.participant_map
excluded_participants = dataset_handler.excluded_participants_metadata

print(f"Raw data directory : {dataset_handler.raw_data_dir}")
print(f"Recordings parsed  : {len(dataset_metadata)}")
print(f"Participant-map rows: {len(participant_map)}")
print(f"Excluded entries   : {len(excluded_participants)}")

In [ ]:
participant_map.head()

## Tidy Metadata

The parsed metadata uses `SingleDataMetadata` enums as columns and enum values as
cell content. We build a plain-string "tidy" table that is easier to group and plot.

In [ ]:
def to_tidy(metadata: pd.DataFrame) -> pd.DataFrame:
    """Convert parsed metadata into a plain-string DataFrame for analysis.

    :param metadata: Parsed dataset metadata (enum columns / enum values).
    :return: DataFrame with string columns: participant, eeg_condition,
        condition, music_type, filename.
    """
    return pd.DataFrame(
        {
            "participant": metadata[SingleDataMetadata.PARTICIPANT_ID].astype(str),
            "eeg_condition": metadata[SingleDataMetadata.EEG_CONDITION_ID].map(
                lambda x: x.value
            ),
            "condition": metadata[SingleDataMetadata.CONDITION].map(lambda x: x.value),
            "music_type": metadata[SingleDataMetadata.MUSIC_TYPE].map(
                lambda x: x.value
            ),
            "filename": metadata[SingleDataMetadata.FILENAME].astype(str),
        }
    )


tidy = to_tidy(dataset_metadata)
print(f"Columns: {list(tidy.columns)}")
tidy.head(20)

## Recordings per Participant

In [ ]:
per_participant = (
    tidy.groupby("participant").size().rename("n_recordings").reset_index()
)
print(f"Distinct participants with data: {per_participant.shape[0]}")
print(per_participant["n_recordings"].describe().to_string())

fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(per_participant)), 4))
sns.barplot(data=per_participant, x="participant", y="n_recordings", ax=ax)
ax.set_title(f"Recordings per participant — {EXPERIMENT.value}")
ax.set_xlabel("Participant")
ax.set_ylabel("# recordings")
plt.xticks(rotation=90)
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "recordings_per_participant.png", dpi=150)
plt.show()
per_participant

## Condition / Music-Type Balance

In [ ]:
pd.crosstab(tidy["condition"], tidy["music_type"], margins=True)

## Presence Matrix

Counts of recordings for every expected `participant × (condition / music_type)` cell.
The grid is reindexed to the **full** set of expected participants (from the participant
mapping) and expected music types, so blanks reveal gaps:

* **0** (red) — missing trial
* **1** (green) — exactly one recording (expected)
* **2+** (orange) — duplicate trials

In [ ]:
expected_music = [m.value for m in EXPERIMENT_MUSIC_TYPES[EXPERIMENT]]

# Expected participants and conditions come from the participant mapping.
map_norm = pd.DataFrame(
    {
        "participant": participant_map["participant"].astype(str).str.replace(
            "PSI", "", regex=False
        ),
        "condition": participant_map["condition"].astype(str),
    }
)
all_participants = sorted(map_norm["participant"].unique())
expected_combos = sorted(
    {
        f"{cond} / {mt}"
        for cond in map_norm["condition"].unique()
        for mt in expected_music
    }
)

tidy = tidy.assign(combo=tidy["condition"] + " / " + tidy["music_type"])
matrix = (
    tidy.pivot_table(
        index="participant",
        columns="combo",
        values="filename",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(index=all_participants, columns=expected_combos, fill_value=0)
    .astype(int)
)

# Colour: 0 -> red (missing), 1 -> green (ok), 2+ -> orange (duplicate).
cmap = ListedColormap(["#d65f5f", "#5fba7d", "#ee9b3a"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 100], cmap.N)

fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(expected_combos)), max(4, 0.32 * len(all_participants))))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap=cmap,
    norm=norm,
    cbar=False,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)
ax.set_title(f"Trial presence matrix — {EXPERIMENT.value}")
ax.set_xlabel("Condition / Music type")
ax.set_ylabel("Participant")
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "presence_matrix.png", dpi=150)
plt.show()

## Duplicate Trials

The same `participant × condition × music_type` combination recorded more than once.
These must be resolved (e.g. via the excluded-participants list) before the data is
stacked for analysis.

In [ ]:
group_keys = ["participant", "condition", "music_type"]
group_sizes = tidy.groupby(group_keys).size().rename("n").reset_index()
duplicate_keys = group_sizes[group_sizes["n"] > 1]

if duplicate_keys.empty:
    print("No duplicate trials found.")
    duplicates = pd.DataFrame(columns=tidy.columns)
else:
    print(f"Found {len(duplicate_keys)} duplicated combination(s):")
    duplicates = (
        tidy.merge(duplicate_keys[group_keys], on=group_keys, how="inner")
        .sort_values(group_keys + ["filename"])
        .reset_index(drop=True)
    )
duplicates

## Missing Trials

Expected `participant × condition × music_type` combinations (derived from the
participant mapping and the experiment's music types) that have **no** recording.

In [ ]:
expected_trials = {
    (row.participant, row.condition, mt)
    for row in map_norm.itertuples()
    for mt in expected_music
}
present_trials = set(
    zip(tidy["participant"], tidy["condition"], tidy["music_type"])
)

missing_trials = sorted(expected_trials - present_trials)
missing_df = pd.DataFrame(
    missing_trials, columns=["participant", "condition", "music_type"]
)
print(f"Expected trials : {len(expected_trials)}")
print(f"Present trials  : {len(present_trials)}")
print(f"Missing trials  : {len(missing_df)}")
missing_df

## Unexpected Trials

Recordings present in the raw data that do **not** match any expected combination
(e.g. an unknown participant, or a condition that the mapping does not list).

In [ ]:
unexpected_trials = sorted(present_trials - expected_trials)
unexpected_df = pd.DataFrame(
    unexpected_trials, columns=["participant", "condition", "music_type"]
)
if unexpected_df.empty:
    print("No unexpected trials — all recordings match an expected combination.")
else:
    print(f"Found {len(unexpected_df)} unexpected combination(s):")
unexpected_df

## Excluded Participants

Recordings flagged for exclusion (e.g. bad data quality) in
`data/excluded_participants/<experiment>.csv`.

In [ ]:
print(f"Excluded entries: {len(excluded_participants)}")
excluded_participants